# 02 — Knowledge Store Workflows Demo (In-Memory Object Flows)

Step-by-step interactive demonstration of every Dual-Engine Knowledge workflow using Python objects held in `InMemoryWorkspace` (zero database setup required):

1. **Hierarchical Tree Indexing** (`TreeIndexerService`)
2. **Material Analysis & Prerequisite Ontology** (`EvaluatorService.analyze_material` + `OntologyService`)
3. **Autonomous Submission Grading & Mastery Tracking** (`EvaluatorService.grade_submission`)
4. **Anti-Drift Audit, Misconception Dreaming & Review** (`ConsolidationService` + `OntologyService`)
5. **Dual-Engine RAG Chat & Soft-Warning Gate** (`ChatService`)

In [ ]:
from pathlib import Path
import sys

for candidate in (Path.cwd(), *Path.cwd().parents, Path.cwd() / "backend"):
    if (candidate / "src" / "main.py").exists():
        if str(candidate) not in sys.path:
            sys.path.insert(0, str(candidate))
        break

from notebooks._common import (
    InMemoryWorkspace,
    banner,
    pretty_print,
    render_mastery_rows,
    render_material_tree,
    render_misconceptions,
    render_prereq_chain,
    render_visualization,
)
from tests._harness import (
    sample_chat_payload,
    sample_grade_request,
    sample_long_curriculum_markdown,
    sample_material_analyze_request,
)

USE_REAL_LLM = False  # Set True to call OpenRouter while keeping all storage in memory

ws = InMemoryWorkspace()
await ws.start(use_real_llm=USE_REAL_LLM)
pretty_print("Initial In-Memory Object Counts", ws.summary())

## Step 1 — Hierarchical Tree Indexing (`TreeIndexerService`)

In [ ]:
from src.service.tree_indexer import TreeIndexerService

curriculum_md = sample_long_curriculum_markdown(min_chars=16000)
sections = TreeIndexerService._extract_markdown_sections(curriculum_md)
print(f"Extracted {len(sections)} Markdown sections ({len(curriculum_md)} chars)")

tree = await TreeIndexerService.build_tree_from_text(
    material_id=ws.material_id,
    material_name="Unit 5: Chemical Kinetics",
    text=curriculum_md,
    model=ws.model,
    api_key="notebook-in-memory",
)
await TreeIndexerService.persist_tree(tree)

loaded_tree = await TreeIndexerService.get_tree(ws.material_id)
traversal = await TreeIndexerService.traverse_tree(ws.material_id, ["rate", "laws"])

banner("STEP 1", "Hierarchical Tree Indexing (TreeIndexerService)")
if loaded_tree:
    render_material_tree(loaded_tree)
pretty_print("Top-Down Traversal Excerpts", traversal.selected_excerpts)
pretty_print("Object Counts", ws.summary())

## Step 2 — Material Analysis & Prerequisite Ontology Graph (`EvaluatorService` + `OntologyService`)

In [ ]:
from src.service.evaluator import EvaluatorService
from src.service.ontology import OntologyService
from src.types.ai import (
    MaterialAnalyzeRequest,
    MaterialConceptMapping,
    OntologyConcept,
    OntologyRelationship,
)

analyze_req = MaterialAnalyzeRequest(
    **sample_material_analyze_request(material_id=ws.material_id, class_id=ws.class_id, long_document=True)
)
analyze_res = await EvaluatorService.analyze_material(analyze_req, ws.model)
pretty_print("MaterialAnalyzeResponse", analyze_res)

# Build an explicit 3-concept prerequisite chain in memory
c_stoich = await OntologyService.upsert_concept(
    OntologyConcept(subject="AP Chemistry", code="CHEM-1.1", name="Stoichiometry", description="Mole conversions", bloom_level="Apply")
)
c_kinetics = await OntologyService.upsert_concept(
    OntologyConcept(subject="AP Chemistry", code="CHEM-5.0", name="Chemical Kinetics", description="Reaction rates", bloom_level="Understand")
)
c_rates = await OntologyService.upsert_concept(
    OntologyConcept(subject="AP Chemistry", code="CHEM-5.1", name="Rate Laws", description="Reaction orders & rate constants", bloom_level="Analyze")
)

await OntologyService.upsert_relationship(
    OntologyRelationship(source_concept_id=c_stoich, target_concept_id=c_kinetics, relationship_type="prerequisite_of")
)
await OntologyService.upsert_relationship(
    OntologyRelationship(source_concept_id=c_kinetics, target_concept_id=c_rates, relationship_type="prerequisite_of")
)
await OntologyService.link_material_to_concepts([
    MaterialConceptMapping(material_id=ws.material_id, concept_id=c_rates, page_number=2, relationship="teaches")
])

prereq_chain = await OntologyService.get_prerequisite_graph_cte([c_rates], max_depth=3)
gate_ok = await OntologyService.check_answerable(ws.class_id, ["chemical", "kinetics", "rate", "laws"])

banner("STEP 2", "Prerequisite Chain & Answerability Gate")
print("Prerequisite Chain for 'Rate Laws':")
render_prereq_chain(prereq_chain)
pretty_print("Pre-Query Answerability Check", gate_ok)

## Step 3 — Autonomous Grading & Student Mastery Sync (`EvaluatorService.grade_submission`)

In [ ]:
from src.types.ai import GradeRequest

grade_responses = []
for sub_id in ws.submission_ids:
    req = GradeRequest(
        **sample_grade_request(
            submission_id=sub_id,
            class_id=ws.class_id,
            student_id=ws.student_id,
            material_id=ws.material_id,
        )
    )
    grade_responses.append(await EvaluatorService.grade_submission(req, ws.model))

banner("STEP 3", "Submission Grading & Student Concept Mastery")
pretty_print("GradeResponse (Submission 1)", grade_responses[0])
mastery_rows = [
    {
        "student_label": "Demo Student",
        "concept_name": ws.db.ontology_concepts.get(r["concept_id"], {}).get("name", r["concept_id"]),
        "mastery_score": r["mastery_score"],
        "status": r["status"],
    }
    for r in ws.db.student_concept_mastery.values()
]
print("In-Memory Student Concept Mastery:")
render_mastery_rows(mastery_rows)

## Step 4 — Anti-Drift Consolidation, Misconception Dreaming & Teacher Review

In [ ]:
import uuid
from src.service.consolidation import ConsolidationService
from src.types.ai import ConsolidationRequest

# Add an orphan concept and a 2-node prerequisite cycle to observe automatic detection & repair
await OntologyService.upsert_concept(
    OntologyConcept(subject="AP Chemistry", name="Orphan Concept", description="Unlinked node")
)
await OntologyService.upsert_relationship(
    OntologyRelationship(source_concept_id=c_rates, target_concept_id=c_kinetics, relationship_type="prerequisite_of")
)

cons_res = await ConsolidationService.run_daily_consolidation(
    ConsolidationRequest(class_id=ws.class_id, run_type="daily", force_dreaming=True),
    ws.model,
)
banner("STEP 4", "Anti-Drift Consolidation & Misconception Review")
pretty_print("ConsolidationResponse", cons_res)

proposed = await OntologyService.get_proposed_misconceptions(class_id=ws.class_id)
print("Proposed Misconceptions:")
render_misconceptions(proposed)

if proposed and proposed[0].id:
    await OntologyService.review_misconception(proposed[0].id, "confirmed", str(uuid.uuid4()))
    remaining = await OntologyService.get_proposed_misconceptions(class_id=ws.class_id)
    print(f"Confirmed misconception {proposed[0].id}; remaining proposed count: {len(remaining)}")

## Step 5 — Dual-Engine RAG Chat (`ChatService`) & Cleanup

In [ ]:
from src.service.chat import ChatService
from src.types.schemas import ChatPayload

# 1. Uncovered topic -> soft-warning caveat without calling LLM
warn_payload = ChatPayload(
    **sample_chat_payload(prompt="Thermodynamics Entropy", class_id=ws.class_id, confirmed_soft_warning=False)
)
warn_reply = await ChatService.process_chat_message(warn_payload, ws.model)
pretty_print("Unverified Topic (Soft Warning)", warn_reply)

# 2. Grounded topic -> fuses PageIndex tree excerpts + prerequisite chains + mastery snapshot
rag_payload = ChatPayload(
    **sample_chat_payload(prompt="Chemical Kinetics and Rate Laws", class_id=ws.class_id, confirmed_soft_warning=False)
)
banner("STEP 5", "Dual-Engine RAG Chat & Visualization")
rag_reply = await ChatService.process_chat_message(rag_payload, ws.model)
pretty_print("Dual-Engine RAG Chat Reply", rag_reply)
viz = rag_reply.get("visualization")
render_visualization(viz if isinstance(viz, dict) else None)

pretty_print("Final In-Memory Object Counts", ws.summary())
await ws.stop()